# FarmHealth NDVI Visualization

Exploratory visualization of the pipeline's three deliverables: the county boundary used as AOI, the mean-NDVI seasonal timeseries, and the monthly NDVI netCDF cube. Run `main.py` first so `data/` and `outputs/` are populated.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Config

config = Config(data_dir=PROJECT_ROOT / "data", out_dir=PROJECT_ROOT / "outputs")
config

## County boundary (VG250 shapefile)

Loads the same BKG VG250 Kreise layer the pipeline uses, and highlights the configured county among all German Kreise.

In [ ]:
kreise_path = next(Path(config.data_dir, "vg250").rglob("*_KRS.shp"))
kreise = gpd.read_file(kreise_path)
if "GF" in kreise.columns and (kreise["GF"] == 4).any():
    kreise = kreise[kreise["GF"] == 4]

key_col = next(c for c in ("AGS", "ARS", "RS") if c in kreise.columns)
is_aoi = kreise[key_col].astype(str).str.startswith(config.aoi_key_prefix)

fig, ax = plt.subplots(figsize=(8, 8))
kreise.plot(ax=ax, color="#eeeeee", edgecolor="#999999", linewidth=0.5)
kreise[is_aoi].plot(ax=ax, color="#2e8b57", edgecolor="black")
ax.set_title(f"{config.aoi_name} (highlighted) among German Kreise")
ax.set_axis_off()

In [ ]:
aoi_gdf = kreise[is_aoi].to_crs(4326)
fig, ax = plt.subplots(figsize=(6, 6))
aoi_gdf.plot(ax=ax, color="#2e8b57", edgecolor="black")
ax.set_title(f"{config.aoi_name} boundary (EPSG:4326)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

## NDVI seasonal timeseries

County mean NDVI per month, from `outputs/ndvi_timeseries.csv`.

In [ ]:
ts = pd.read_csv(config.timeseries_csv, parse_dates=["date"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts["date"], ts["ndvi"], marker="o", color="#2e8b57")
ax.set_ylim(-0.1, 1.0)
ax.set_title(f"Mean NDVI \u2014 {config.aoi_name}")
ax.set_xlabel("Month")
ax.set_ylabel("Mean NDVI")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
ts

## NDVI monthly cube (netCDF)

The per-pixel cube from `outputs/ndvi_monthly_*.nc`. The data variable is named `var` in the file; we rename it to `NDVI` for readability. Values are clipped to `[-0.1, 1.0]` for display — a handful of edge pixels fall outside the valid NDVI range due to cloud-gap interpolation artifacts.

In [ ]:
cube = xr.open_dataset(config.netcdf_path)["var"].rename("NDVI")
cube

### All months, faceted

In [ ]:
n_months = cube.sizes["t"]
ncols = 4
nrows = -(-n_months // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
mappable = None
for ax, date in zip(axes.flat, cube["t"].values):
    mappable = cube.sel(t=date).plot.imshow(ax=ax, cmap="RdYlGn", vmin=-0.1, vmax=1.0, add_colorbar=False)
    ax.set_title(pd.Timestamp(date).strftime("%Y-%m"))
    ax.set_xlabel("")
    ax.set_ylabel("")
for ax in axes.flat[n_months:]:
    ax.set_visible(False)
fig.colorbar(mappable, ax=axes, label="NDVI", shrink=0.6)

### Single month, with AOI boundary overlay

In [ ]:
month = cube["t"].values[len(cube["t"]) // 2]
aoi_utm = aoi_gdf.to_crs(config.epsg)

fig, ax = plt.subplots(figsize=(8, 7))
cube.sel(t=month).plot(ax=ax, cmap="RdYlGn", vmin=-0.1, vmax=1.0, cbar_kwargs={"label": "NDVI"})
aoi_utm.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title(f"NDVI — {pd.Timestamp(month).strftime('%B %Y')}")